# 01 · Outcome ladder and count audit: maternal death and timing of SMM — Maternal death (M1) → severe maternal morbidity (M2), MIMIC-IV

| Field | Content |
|---|---|
| Notebook sequence | **01: S1 · S2 · S3 · S4 · G1** → 02: S5 · G2 → 03: S6 · S7 · G3 · R1 · S8 → 04: S8 → 05: S9 (S7) → 06: S9 (S3) → 07: S9 · C1-C3 · R2 · R4 → 08: S9 · C4 → 09: S9 · C4 → 10: G4 · S10 · S11 |
| What it does | Fixes the outcome ladder and counts the deaths of the ideal outcome, maternal death (M1). For the next rung, SMM (M2): SMM is defined for the whole admission and diagnosis codes carry no time, but procedure codes carry a **date**. This notebook counts how many SMM events can be placed before, around or after the prediction point T = admission + 24 h. It trains no model and only reads the database |
| Stages and gates | S1 governance · S2 universe and population · S3 outcome definition and event timing · S4 count audit · G1 · origin of the landmark rule and of criterion C1 |
| Inputs | Local PostgreSQL MIMIC-IV database (tables of `sql/`): `predictors_24h_v2`, `procedures_icd`, `smm_code_patterns_v2` |
| Outputs | `data/outputs_smm_timing.txt` |
| Data use | Only aggregate results are shown and saved; nothing at patient level. Do not add cells that show or export rows |
| Time | Seconds to a few minutes |
| Next | `02_patient_overlap.ipynb` |
| Spanish version | `01_smm_timing_vs_prediction_time_es.ipynb` (same code) |

| Timing | Rule (D = calendar date of the procedure) | Meaning |
|---|---|---|
| `before` | `D + 1 day <= T` | the whole day ended before T |
| `after` | `D >= T` | the day began at or after T |
| `ambiguous` | otherwise | the day contains T; the order cannot be determined |

Each SMM admission gets **one** category: **A** a procedure indicator certainly before T; **B** none before but one ambiguous; **C** procedure indicators only after T; **D** no procedure indicator (diagnosis codes only; no time).

## S1 · Governance and configuration — libraries and connection

In [ ]:
import os, sys, io, contextlib
import pandas as pd
from sqlalchemy import text

# notebooks/ -> project root is one level up
sys.path.append(os.path.abspath(os.path.join("..")))
import libraries.database as db_lib
engine = db_lib.get_connect_database()

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

EXPECTED = {"eligible_smm": 445, "excluded_deliveries": 42, "excluded_smm": 2, "excluded_deaths": 1, "deaths_obstetric": 10, "deaths_before_T": 5}

def q(sql):
    # text() is required: the queries contain a literal percent sign (LIKE ... || '%'), which the
    # PostgreSQL driver would otherwise read as a parameter placeholder ("immutabledict is not a sequence").
    return pd.read_sql(text(sql), engine)

# ----------------------------------------------------------------------------- language and presentation
# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
# Nothing below changes a computation: it only decides how tables and messages are shown. The English aggregate
# reports data/outputs_*.txt are written exactly as before (the article cites them); the Spanish notebook also
# writes a translated copy, data/outputs_*_es.txt.
LANG = "en"
import builtins, contextlib, importlib, io, html as _html
import numpy as np
import libraries.i18n_es, libraries.i18n          # reloaded so that a running kernel picks up dictionary changes
importlib.reload(libraries.i18n_es); importlib.reload(libraries.i18n)
from libraries.i18n import translate
pd.set_option("display.max_colwidth", None)
def _tr(s, lang=None):
    """Translate one line keeping its indentation and trailing spaces."""
    lang = lang or LANG
    if lang != "es" or not isinstance(s, str):
        return s
    out = []
    for ln in s.split("\n"):
        core = ln.strip()
        if not core:
            out.append(ln); continue
        i = ln.index(core)
        out.append(ln[:i] + translate(core, "es") + ln[i + len(core):])
    return "\n".join(out)

_REC = None          # while emit() runs: the blocks of the English report, [("text", line) | ("table", frame, index)]

def print(*args, sep=" ", end="\n", file=None, flush=False):
    """Inside emit() the English text goes to the report unchanged; outside it, messages follow LANG."""
    if _REC is not None and file is None:
        _REC.append(("text", sep.join(str(a) for a in args)))
        return builtins.print(*args, sep=sep, end=end, flush=flush)
    if LANG == "es":
        args = [_tr(a, "es") for a in args]
    return builtins.print(*args, sep=sep, end=end, file=file, flush=flush)

def _cell_fmt(v):
    # Display format of a cell: thousands separator for counts, 4 significant digits for small values.
    if isinstance(v, (bool, np.bool_)) or v is None or (isinstance(v, float) and np.isnan(v)):
        return v
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        if float(v).is_integer() and abs(v) >= 1000:
            return f"{int(v):,}"
        return f"{v:.4g}" if abs(v) < 1 else f"{v:,.4f}".rstrip("0").rstrip(".")
    return v

def _html_table(df, index):
    df = df.astype(object)
    df = df.map(_cell_fmt) if hasattr(df, "map") else df.applymap(_cell_fmt)
    return f"<div style='max-height:420px; overflow:auto'>{df.to_html(index=index, na_rep='', border=0)}</div>"

def _in_notebook():
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except ImportError:
        return False

def table(df, index=False, title=None):
    """Show a table. Inside emit() it also prints the English text of the report (df.to_string)."""
    if isinstance(df, pd.Series):
        df = df.to_frame()
        index = True
    if _REC is not None:
        _REC.append(("table", df, index))
        return builtins.print(df.to_string(index=index))
    if not _in_notebook():
        return builtins.print(translate(df, LANG).to_string(index=index))
    from IPython.display import HTML, display as _display
    head = f"<p style='margin:14px 0 4px 0'><b>{_html.escape(_tr(title))}</b></p>" if title else ""
    _display(HTML(head + _html_table(translate(df, LANG), index)))

def _blocks_html(blocks):
    """Report blocks as HTML: a text line right before a table is its title; rules of '=' are dropped."""
    out, pending = [], []
    def flush_text():
        for ln in pending:
            if ln.strip():
                out.append(f"<div style='white-space:pre-wrap; font-family:monospace; margin:2px 0'>{_html.escape(_tr(ln.rstrip()))}</div>")
        pending.clear()
    for b in blocks:
        if b[0] == "text":
            pending.extend(l for l in b[1].split("\n") if not set(l.strip()) <= {"="} or not l.strip())
        else:
            title = None
            while pending and not pending[-1].strip():
                pending.pop()
            if pending:
                title = pending.pop().strip()
            flush_text()
            if title:
                out.append(f"<p style='margin:14px 0 4px 0'><b>{_html.escape(_tr(title))}</b></p>")
            out.append(_html_table(translate(b[1], LANG), b[2]))
    flush_text()
    return "\n".join(out)

def _blocks_text_es(blocks):
    """Spanish copy of a report (data/outputs_*_es.txt)."""
    lines = []
    for b in blocks:
        if b[0] == "text":
            lines.extend(_tr(l, "es") for l in b[1].split("\n"))
        else:
            lines.extend(translate(b[1], "es").to_string(index=b[2]).split("\n"))
    return "\n".join(lines) + "\n"

def _emit_to(path, fn, show=True):
    """Run fn, write its English text to `path` exactly as printed and, from the Spanish notebook, a translated copy
    (*_es.txt); show the report as tables."""
    global _REC
    buf = io.StringIO()
    _REC = []
    try:
        with contextlib.redirect_stdout(buf):
            fn()
    finally:
        blocks, _REC = _REC, None
    text = buf.getvalue()
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(text)
    if show:
        show_report(blocks)
    if LANG == "es":
        with open(path.replace(".txt", "_es.txt"), "w", encoding="utf-8") as fh:
            fh.write(_blocks_text_es(blocks))
    return text

def show_report(blocks):
    if _in_notebook():
        from IPython.display import HTML, display as _display
        _display(HTML(_blocks_html(blocks)))
    else:
        for b in blocks:
            builtins.print(_tr(b[1]) if b[0] == "text" else translate(b[1], LANG).to_string(index=b[2]))

def notebook_summary(rows, title="Notebook summary"):
    """Summary of the notebook (as in the neonatal repository): one value per row; a value that cannot be computed
    (for example, an analysis that was not run in this session) is shown as n/a."""
    out = {}
    for label, fn in rows:
        try:
            v = fn() if callable(fn) else fn
        except Exception:
            v = "n/a"
        out[label] = v
    table(pd.Series(out, name="value").to_frame(), index=True, title=title)

def _m(res, model, met, d=3):
    """Point value of a metric with its 95% bootstrap interval, from a result with "point" and "boot"."""
    v = f"{res['point'][(model, met)]:.{d}f}"
    b = res.get("boot") or {}
    a = b.get((model, met)) if isinstance(b, dict) else None
    if a is not None and len(a) >= 20:
        lo, hi = np.nanpercentile(a, [2.5, 97.5])
        v += f" ({lo:.{d}f}-{hi:.{d}f})"
    return v

def _cv_auroc(res, model):
    """AUROC averaged over the repeats of a cross-validation result that only keeps out-of-fold predictions."""
    from sklearn.metrics import roc_auc_score
    y = res["y"]
    return float(np.mean([roc_auc_score(y, res["preds"][model][r]) for r in range(len(res["preds"][model]))]))


## S1 · Governance and configuration — source, data use agreement, seed, software and publication rule

Fixed before running any analysis: the data source and its version, the data use agreement, the random seed, the software versions, the reference counts to be reproduced and the publication rule. The cell shows that configuration and checks that the database answers and the packages are present.

| Result | What is done | In this model |
|---|---|---|
| The database answers and the packages are present (ok) | On to S3 | PostgreSQL copy of MIMIC-IV v3.1 |
| No connection to the database | Check `.env` and that PostgreSQL is running (README, steps 2 to 4) | — |
| A package is missing | `pip install -r requirements.txt` with the virtual environment active | — |

In [ ]:
# Stage S1 check: the configuration every notebook of this repository uses, shown so it can be audited.
import platform, importlib
from sqlalchemy import text as _sql_text
def version(module):
    try:
        return importlib.import_module(module).__version__
    except Exception:
        return "NOT INSTALLED"
try:
    with engine.connect() as _con:
        _con.execute(_sql_text("SELECT 1"))
    db_status = "ok"
except Exception:
    db_status = "NO CONNECTION: check .env and that PostgreSQL is running"
rows = [("data source", "MIMIC-IV v3.1 (PhysioNet), Beth Israel Deaconess Medical Center, 2008-2022; dates shifted per patient", "fixed"),
        ("database", "local PostgreSQL copy of MIMIC-IV (schema mimiciv); credentials in .env (not versioned)", db_status),
        ("data use", "PhysioNet Credentialed Health Data Use Agreement and CITI training: research only; no re-identification; "
                     "data not shared or redistributed", "rule"),
        ("publication rule", "only aggregate outputs (data/outputs_*.txt); cells below 10 suppressed; no subject_id, hadm_id "
                             "or stay_id published", "rule"),
        ("random seed", "20260101 (notebooks 03 to 10); notebook 02 replicates the first-version splits with seed 42", "fixed"),
        ("reference counts", "frozen at the count audit (19,383 eligible deliveries, 445 with SMM) and reproduced by gate G1 "
                             "in notebooks 03 to 10", "fixed"),
        ("python", platform.python_version(), "info")]
rows += [(m, version(m), "ok" if version(m) != "NOT INSTALLED" else "INSTALL (requirements.txt)")
         for m in ["pandas", "numpy", "sklearn", "xgboost", "imblearn", "sqlalchemy", "matplotlib"]]
config = pd.DataFrame(rows, columns=["item", "value", "status"]).set_index("item")
table(config, index=True, title="S1 · Configuration of this repository")

## S3 · S4 · G1 — ideal outcome: maternal death (M1)

The outcome card fixes an outcome ladder before any model is fitted. The first rung is the ideal outcome, maternal death in the same admission (M1); the second is severe maternal morbidity (M2), the rung that return R1 takes if gate G3 does not allow a model of M1. This section counts the deaths (aggregates only) and checks them against the reference counts; gate G3 is applied in notebook 03.

| Result | What is done | In this model |
|---|---|---|
| The death counts reproduce the reference (G1 passes) | M1 goes on to gate G3 (notebook 03) | 3 deaths among the eligible deliveries |
| A count differs | Review the tables of `sql/` before going on | — |

In [ ]:
# Stages S3 and S4 for the ideal outcome, maternal death (M1): the outcome ladder and the death counts (aggregates only).
ladder = pd.DataFrame([
    ("1 (ideal)", "maternal death (M1)", "death in the same admission", "S3 and S4 in this notebook; gate G3 in notebook 03"),
    ("2", "severe maternal morbidity (M2)", "CDC indicators in the same admission (transfusion excluded)", "return R1; notebooks 03 to 10"),
    ("return R2", "escalation of care after 24 h", "ICU admission, invasive ventilation, an SMM procedure or death after T", "notebook 07"),
], columns=["rung", "outcome", "definition", "where"])
table(ladder, title="S3 · Outcome ladder of the outcome card")

_flag = lambda c: f"CAST({c} AS int) = 1"
m1 = pd.read_sql(text(f"""
    SELECT SUM(CASE WHEN {_flag('eligible_24h')} AND {_flag('is_delivery_v2')} THEN 1 ELSE 0 END) AS eligible_deliveries,
           SUM(CASE WHEN {_flag('eligible_24h')} AND {_flag('died_same_admission')} THEN 1 ELSE 0 END) AS deaths_eligible_obstetric,
           SUM(CASE WHEN {_flag('eligible_24h')} AND {_flag('is_delivery_v2')} AND {_flag('died_same_admission')} THEN 1 ELSE 0 END)
               AS deaths_eligible_deliveries
    FROM mimiciv.predictors_24h_v2"""), engine)
EXPECTED_M1 = {"eligible_deliveries": 19383, "deaths_eligible_obstetric": 5, "deaths_eligible_deliveries": 3}
m1_chk = pd.DataFrame([{"count": k, "loaded": int(m1[k].iloc[0] or 0), "expected": v,
                        "result": "reproduced" if int(m1[k].iloc[0] or 0) == v else "DIFFERS"} for k, v in EXPECTED_M1.items()])
table(m1_chk, title="S4 · G1 — maternal death (M1): reference counts")

## S3 · S4 — next rung, SMM (M2): classification of the procedure-based SMM indicators (admission level)

One row per procedure-based indicator hit with its timing, and one row per admission with its category; only the aggregated tables are fetched.

| Result | What is done | In this model |
|---|---|---|
| Many events are already present at T (category A) | The full label mixes prediction and recognition: landmark population and criterion C1 (notebook 07) | At least a third of the SMM admissions had a procedure before T |
| Few events before T | The full label can be read as prediction | — |

In [ ]:
BASE = """
WITH base AS (
    SELECT hadm_id, admittime, admittime + INTERVAL '24 hours' AS t_pred
    FROM mimiciv.predictors_24h_v2
    WHERE eligible_24h AND is_delivery_v2 AND smm_any
),
pr AS (
    SELECT b.hadm_id, p.indicator_name,
           CASE WHEN CAST(r.chartdate AS TIMESTAMP) + INTERVAL '1 day' <= b.t_pred THEN 'before'
                WHEN CAST(r.chartdate AS TIMESTAMP) >= b.t_pred THEN 'after'
                ELSE 'ambiguous' END AS timing
    FROM base b
    JOIN mimiciv.procedures_icd r ON r.hadm_id = b.hadm_id
    JOIN mimiciv.smm_code_patterns_v2 p
      ON p.code_type = 'PR' AND p.icd_version = r.icd_version
     AND ( (p.match_type = 'PREFIX' AND r.icd_code LIKE p.code_pattern || '%')
        OR (p.match_type = 'EXACT'  AND r.icd_code = p.code_pattern) )
),
dx AS (
    SELECT DISTINCT b.hadm_id
    FROM base b
    JOIN mimiciv.diagnoses_icd d ON d.hadm_id = b.hadm_id
    JOIN mimiciv.smm_code_patterns_v2 p
      ON p.code_type = 'DX' AND p.icd_version = d.icd_version
     AND ( (p.match_type = 'PREFIX' AND d.icd_code LIKE p.code_pattern || '%')
        OR (p.match_type = 'EXACT'  AND d.icd_code = p.code_pattern) )
),
adm AS (
    SELECT b.hadm_id,
           COUNT(pr.timing)                                          AS n_pr,
           SUM(CASE WHEN pr.timing = 'before'    THEN 1 ELSE 0 END)  AS n_before,
           SUM(CASE WHEN pr.timing = 'ambiguous' THEN 1 ELSE 0 END)  AS n_amb,
           SUM(CASE WHEN pr.timing = 'after'     THEN 1 ELSE 0 END)  AS n_after,
           CASE WHEN dx.hadm_id IS NULL THEN 0 ELSE 1 END            AS has_dx
    FROM base b
    LEFT JOIN pr ON pr.hadm_id = b.hadm_id
    LEFT JOIN dx ON dx.hadm_id = b.hadm_id
    GROUP BY b.hadm_id, dx.hadm_id
)
"""

SQL_CATEGORY = BASE + """
SELECT category, has_dx, COUNT(*) AS admissions
FROM (
    SELECT has_dx,
           CASE WHEN n_pr = 0     THEN 'D. no procedure indicator (diagnosis codes only; no time)'
                WHEN n_before > 0 THEN 'A. a procedure indicator certainly before T'
                WHEN n_amb    > 0 THEN 'B. none certainly before T, at least one ambiguous'
                ELSE                   'C. procedure indicators, all certainly after T' END AS category
    FROM adm
) t
GROUP BY category, has_dx
ORDER BY category, has_dx
"""

SQL_INDICATOR = BASE + """
SELECT indicator_name, timing, COUNT(DISTINCT hadm_id) AS admissions
FROM pr
GROUP BY indicator_name, timing
ORDER BY indicator_name, timing
"""

SQL_INDICATOR_DISTINCT = BASE + """
SELECT indicator_name, COUNT(DISTINCT hadm_id) AS distinct_admissions
FROM pr
GROUP BY indicator_name
"""

SQL_TOTAL = BASE + "SELECT COUNT(*) AS smm_admissions FROM adm"

In [ ]:
total = int(q(SQL_TOTAL).smm_admissions.iloc[0])
cat = q(SQL_CATEGORY)
cat["has_dx"] = cat["has_dx"].astype(int)
ind = q(SQL_INDICATOR)
ind_distinct = q(SQL_INDICATOR_DISTINCT).set_index("indicator_name")["distinct_admissions"]

def report_category():
    print("=" * 110)
    print(f"SMM admissions among eligible delivery hospitalizations: {total}  (expected {EXPECTED['eligible_smm']}) "
          f"[{'OK' if total == EXPECTED['eligible_smm'] else 'CHECK: differs from the audit counts'}]")
    print("T = admission time + 24 h; timing from the calendar date of each procedure-based SMM indicator (see the method table)")
    print("=" * 110)
    wide = cat.pivot_table(index="category", columns="has_dx", values="admissions", aggfunc="sum", fill_value=0)
    wide = wide.rename(columns={0: "no diagnosis-based indicator", 1: "with a diagnosis-based indicator"})
    wide["total"] = wide.sum(axis=1)
    wide["% of SMM"] = (100 * wide["total"] / total).round(1)
    print("\n[1] Category of every SMM admission (A to D); the last columns split by presence of a diagnosis-based indicator")
    table(wide, index=True)
    print(f"    Sum of categories: {int(wide['total'].sum())}  [{'OK' if int(wide['total'].sum()) == total else 'CHECK'}]")
    a = int(wide.loc[wide.index.str.startswith("A"), "total"].sum()) if wide.index.str.startswith("A").any() else 0
    b = int(wide.loc[wide.index.str.startswith("B"), "total"].sum()) if wide.index.str.startswith("B").any() else 0
    c = int(wide.loc[wide.index.str.startswith("C"), "total"].sum()) if wide.index.str.startswith("C").any() else 0
    d_ = int(wide.loc[wide.index.str.startswith("D"), "total"].sum()) if wide.index.str.startswith("D").any() else 0
    print("\n    Reading: at least %d SMM admissions (%.1f%%) had a procedure indicator certainly before T; %d more (%.1f%%) cannot be ordered;"
          % (a, 100 * a / total, b, 100 * b / total))
    print("    %d (%.1f%%) have procedure indicators only after T; %d (%.1f%%) rest on diagnosis codes alone and cannot be placed in time."
          % (c, 100 * c / total, d_, 100 * d_ / total))
    pw = ind.pivot_table(index="indicator_name", columns="timing", values="admissions", aggfunc="sum", fill_value=0)
    for t in ("before", "ambiguous", "after"):
        if t not in pw.columns:
            pw[t] = 0
    pw = pw[["before", "ambiguous", "after"]]
    pw["distinct admissions with the indicator"] = ind_distinct.reindex(pw.index).astype(int)
    print("\n[2] Procedure-based indicators: admissions by timing of the indicator (an admission with the same indicator on different days can appear in")
    print("    several of the first three columns, so they need not add up to the last column, which counts each admission once)")
    table(pw.sort_values("distinct admissions with the indicator", ascending=False), index=True)

report_category()

## S2 · S4 · G1 — consistency checks with the reference counts

(a) Delivery hospitalizations excluded because the stay ended before 24 h; (b) timing of the in-hospital deaths of the obstetric universe relative to T. Each line ends with [OK] when it reproduces the reference.

In [ ]:
SQL_EXCL = """
SELECT COUNT(*) AS n,
       SUM(CASE WHEN smm_any THEN 1 ELSE 0 END) AS smm,
       SUM(CASE WHEN died_same_admission THEN 1 ELSE 0 END) AS deaths
FROM mimiciv.predictors_24h_v2
WHERE is_delivery_v2 AND NOT COALESCE(eligible_24h, FALSE)
"""
SQL_DEATH = """
SELECT CASE WHEN a.deathtime IS NULL THEN 'death without recorded time'
            WHEN a.deathtime <  p.admittime + INTERVAL '24 hours' THEN 'before T (stay ended before the prediction point)'
            ELSE 'at or after T' END AS timing,
       CASE WHEN p.is_delivery_v2 THEN 'delivery' ELSE 'other obstetric' END AS admission_type,
       COUNT(*) AS deaths
FROM mimiciv.predictors_24h_v2 p
JOIN mimiciv.admissions a ON a.hadm_id = p.hadm_id
WHERE p.died_same_admission
GROUP BY 1, 2
ORDER BY 1, 2
"""
ex = q(SQL_EXCL)
dt = q(SQL_DEATH)

def report_checks():
    print("=" * 110)
    print("CONSISTENCY CHECKS")
    print("=" * 110)
    n, s, d = int(ex.n.iloc[0]), int(ex.smm.iloc[0] or 0), int(ex.deaths.iloc[0] or 0)
    print(f"\n[3] Delivery hospitalizations not eligible at 24 h: {n} (expected {EXPECTED['excluded_deliveries']}), "
          f"with SMM {s} (expected {EXPECTED['excluded_smm']}), with death {d} (expected {EXPECTED['excluded_deaths']})  "
          f"[{'OK' if (n, s, d) == (EXPECTED['excluded_deliveries'], EXPECTED['excluded_smm'], EXPECTED['excluded_deaths']) else 'CHECK'}]")
    print("\n[4] In-hospital deaths of the obstetric universe by timing relative to T")
    table(dt)
    tot = int(dt.deaths.sum()); bef = int(dt.loc[dt.timing.str.startswith("before"), "deaths"].sum())
    print(f"    Deaths: {tot} (expected {EXPECTED['deaths_obstetric']}); before T: {bef} (expected {EXPECTED['deaths_before_T']})  "
          f"[{'OK' if (tot, bef) == (EXPECTED['deaths_obstetric'], EXPECTED['deaths_before_T']) else 'CHECK'}]")

report_checks()

## S11 · Reporting — aggregate report

Writes `data/outputs_smm_timing.txt` (the tables shown above; counts only).

In [ ]:
def _timing_report():
    report_category()
    print()
    report_checks()

# Same English text as before in data/outputs_smm_timing.txt (and, from the Spanish notebook, *_es.txt)
text = _emit_to("../data/outputs_smm_timing.txt", _timing_report, show=False)
print(f"[aggregate report saved as data/outputs_smm_timing.txt  ({len(text)} characters)]")

## Notebook summary

The ideal outcome (M1), the next rung (M2) and the timing of the SMM evidence relative to T.

In [ ]:
# Summary of notebook 01
def _cat(letter):
    return int(cat.loc[cat.category.str.startswith(letter), "admissions"].sum())
notebook_summary([
    ("maternal deaths among eligible deliveries (M1)", lambda: int(m1["deaths_eligible_deliveries"].iloc[0])),
    ("gate G1, maternal death counts", lambda: "passes" if (m1_chk["result"] == "reproduced").all() else "does NOT pass"),
    ("next step for M1", "gate G3 in notebook 03"),
    ("SMM admissions among eligible deliveries (M2)", lambda: total),
    ("SMM with a procedure certainly before T (category A)", lambda: f"{_cat('A')} ({100 * _cat('A') / total:.1f}%)"),
    ("SMM that cannot be ordered (category B)", lambda: f"{_cat('B')} ({100 * _cat('B') / total:.1f}%)"),
    ("SMM placed only after T (category C)", lambda: f"{_cat('C')} ({100 * _cat('C') / total:.1f}%)"),
    ("SMM on diagnosis codes only, no time (category D)", lambda: f"{_cat('D')} ({100 * _cat('D') / total:.1f}%)"),
    ("gate G1, consistency checks", lambda: "passes" if (
        (int(ex.n.iloc[0]), int(ex.smm.iloc[0] or 0), int(ex.deaths.iloc[0] or 0))
        == (EXPECTED["excluded_deliveries"], EXPECTED["excluded_smm"], EXPECTED["excluded_deaths"])
        and int(dt.deaths.sum()) == EXPECTED["deaths_obstetric"]) else "does NOT pass"),
    ("consequence for the framework", "part of the SMM is already present at T: landmark population and criterion C1 (notebook 07)"),
])